# Build Vector Index on Colab Pro GPU

Run this notebook on Colab Pro (T4 or better) to:
1. Clone your repo
2. Download FDA Orange Book
3. Build the ChromaDB index with bge-m3 on GPU (~30 sec vs ~25 min on CPU)
4. Save the ChromaDB folder to Google Drive
5. Download it to your local machine for fast querying

**Prerequisites:**
- Runtime → Change runtime type → T4 GPU
- Google Drive mounted

## 1. Mount Google Drive + clone repo

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Edit this path to where you want ChromaDB saved on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/pharma-intelligence/chroma_db'

REPO_URL = 'https://github.com/siriponsri/pharma-intelligence.git'
REPO_DIR = '/content/pharma-intelligence'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists, reusing local copy.')

%cd {REPO_DIR}

Mounted at /content/drive
Cloning into '/content/pharma-intelligence'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 77 (delta 10), reused 76 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 71.05 KiB | 1.08 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/pharma-intelligence


## 2. Install dependencies

In [2]:
# Install into the current notebook kernel
%pip install -q -U pip setuptools wheel jedi
%pip install -q -e ".[notebook]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pharma-intel (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.2.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client 8.8.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.5.5 which is inco

In [3]:
import sys
from pathlib import Path

repo_dir = Path("/content/pharma-intelligence")
src_dir = repo_dir / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pharma_intel
print("Python executable:", sys.executable)
print("pharma_intel imported from:", pharma_intel.__file__)

Python executable: /usr/bin/python3
pharma_intel imported from: /content/pharma-intelligence/src/pharma_intel/__init__.py


## 3. Set env vars (ThaiLLM API key)

⚠️ Do NOT commit the key. Use Colab's "Secrets" feature (🔑 icon in left sidebar) to store `THAILLM_API_KEY`, then load here:

In [4]:
import os
from google.colab import userdata

os.environ['THAILLM_API_KEY'] = userdata.get('THAILLM_API_KEY')
os.environ['LLM_PROVIDER'] = 'thaillm'

# Save ChromaDB directly on Drive
os.environ['CHROMA_PERSIST_DIR'] = DRIVE_OUTPUT
os.environ['DATA_DIR'] = '/content/drive/MyDrive/pharma-intel/data'
os.environ['MODEL_CACHE_DIR'] = '/content/drive/MyDrive/pharma-intel/.cache/models'

## 4. Verify GPU is available

In [5]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

CUDA available: True
Device: Tesla T4


## 5. Run the pipeline

In [6]:
from pharma_intel.ingestion.fda_orange_book import run_pipeline
from pharma_intel.rag.indexer import index_monographs

# Step 1: download + parse + filter
monographs = run_pipeline(force_download=False)
print(f'\n✓ {len(monographs)} monographs ready for indexing')

2026-04-21 14:13:13 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:136 - Downloading Orange Book from https://www.fda.gov/media/76860/download...
2026-04-21 14:13:13 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:140 - Downloaded 1.0 MB
2026-04-21 14:13:13 | INFO     | pharma_intel.ingestion.fda_orange_book:download_orange_book:144 - Extracted files: ['exclusivity.txt', 'patent.txt', 'products.txt']
2026-04-21 14:13:14 | INFO     | pharma_intel.ingestion.fda_orange_book:load_products:187 - Loaded 48083 product records from products.txt
2026-04-21 14:13:14 | INFO     | pharma_intel.ingestion.fda_orange_book:load_patents:194 - Loaded 20858 patent records from patent.txt
2026-04-21 14:13:14 | INFO     | pharma_intel.ingestion.fda_orange_book:load_exclusivity:201 - Loaded 2054 exclusivity records from exclusivity.txt
2026-04-21 14:13:14 | INFO     | pharma_intel.ingestion.fda_orange_book:filter_cardiometabolic:228 - Filtered to 5479 cardi


✓ 5479 monographs ready for indexing


In [7]:
# Step 2: embed with bge-m3 on GPU — this is the speedup
import time
start = time.time()
store = index_monographs(monographs, batch_size=64)   # larger batch on GPU
print(f'\n✓ Indexing took {time.time() - start:.1f} seconds')
print(f'✓ Collection size: {store.collection.count()}')

2026-04-21 14:13:45 | INFO     | pharma_intel.rag.indexer:index_monographs:23 - Indexing 5479 monographs into 'cardiometabolic_drugs'
2026-04-21 14:13:45 | INFO     | pharma_intel.rag.embeddings:get_embedder:20 - Loading embedding model: BAAI/bge-m3
2026-04-21 14:13:45 | INFO     | pharma_intel.rag.embeddings:get_embedder:21 - First run will download the model (~2GB for bge-m3). Subsequent runs use cache.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/content/pharma-intelligence/src/pharma_intel/rag/embeddings.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.info(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
2026-04-21 14:14:48 | INFO     | pharma_intel.rag.embeddings:get_embedder:28 - Embedding dim: 1024
2026-04-21 14:14:48 | INFO     | pharma_intel.rag.embeddings:embed_texts:35 - Embedding 5479 texts (batch_size=64)...


Batches:   0%|          | 0/86 [00:00<?, ?it/s]

2026-04-21 14:17:25 | INFO     | pharma_intel.rag.vectorstore:__init__:39 - VectorStore ready — collection='cardiometabolic_drugs', count=0


InternalError: ValueError: Batch size of 5479 is greater than max batch size of 5461

## 6. Test query on Colab

In [ ]:
from pharma_intel.rag import RAGEngine

engine = RAGEngine(top_k=5)
result = engine.answer('What patents cover empagliflozin and when do they expire?')

print('--- ANSWER ---')
print(result.answer)
print(f'\n--- Retrieved {len(result.retrieved)} sources ---')
for c in result.retrieved:
    print(f'  [{c.doc_id}] {c.metadata["ingredient"]} — score={c.score:.3f}')

## 7. ChromaDB is already on Drive — pull to local

Since we set `CHROMA_PERSIST_DIR` to the Drive path, the index is already there.

On your local machine, mount Drive (e.g., with Google Drive Desktop) or download
the `chroma_db/` folder, then run locally:

```bash
export CHROMA_PERSIST_DIR=/path/to/downloaded/chroma_db
python scripts/query_rag.py 'your question'
```

Local querying is fast (<1 sec per query) because only 1 embedding is computed per query.